# పాఠం 02 - మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ అన్వేషణ

**Microsoft Agent Framework (MAF)** అనేది AI ఏజెంట్లను నిర్మించడానికి ఒక ఏకీకృత ఫ్రేమ్‌వర్క్. ఇది నాలుగు ప్రాథమిక నిర్మాణ 블ాక్లు ఉన్న శుభ్రంగా, నియమించదగిన వాస్తవ నిర్మాణాన్ని అందిస్తుంది:

- **క్లయింట్** – AI మోడల్ ఎండ్పాయింట్‌కి కనెక్ట్ అవుతుంది మరియు కమ్యూనికేషన్‌ను నిర్వహిస్తుంది
- **ఏజెంట్** – క్లయింట్‌ను సూచనలతో మరియు టూల్ నిర్వచనాలతో చుట్టుముట్టుతుంది
- **టూల్స్** – ఏజెంట్ సామర్థ్యాలను మోడల్ కాల్ చేయగల కస్టమ్ ఫంక్షన్లతో విస్తరిస్తాయి
- **సెషన్** – బహుళ సార్లు సంభాషణ చరిత్రను నిర్వహిస్తుంది

ఈ పాఠంలో, మనం ఈ భావనలను ఉపయోగించి గమ్యస్థాన అందుబాటుని తనిఖీ చేసే **ప్ర‌వాణి బుకింగ్ ఏజెంట్** ను నిర్మిస్తాము.


## సెటప్


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ఏజెంట్ ఫ్రేమ్‌వర్క్ ఆర్కిటెక్చర్‌ను అర్థం చేసుకోవడం

మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఒక లేయర్డ్ ఆర్కిటెక్చర్‌ను అనుసరిస్తుంది:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **క్లయింట్** – ఒక `FoundryChatClient` Azure OpenAI డిప్లాయ్‌మెంట్‌కు కనెక్ట్ అవుతుంది. ఇది ఆథెంటికేషన్, అభ్యర్థన ఫార్మాటింగ్, మరియు స్పందన పార్సింగ్ నిర్వహిస్తుంది.
2. **ఏజెంట్** – క్లయింట్ ద్వారా `provider.create_agent()` ద్వారా సృష్టించబడుతుంది, ఏజెంట్ మోడల్ యాక్సెస్, సూచనలు (సిస్టమ్ ప్రాంప్ట్) మరియు టూల్స్‌ను కలిపి పనిచేస్తుంది.
3. **టూల్స్** – `@tool` కొలుచుకున్న పాథాన్ ఫంక్షన్లు, వీటిని ఏజెంట్ చర్యలు చేపట్టడానికి లేదా డేటా పొందడానికి కాల్ చేయవచ్చు.
4. **సెషన్** – `AgentSession` ఆబ్జెక్ట్ (`agent.create_session()` ద్వారా సృష్టించబడుతుంది) సంభాషణ చరిత్రను నిల్వ చేస్తుంది, ఇది ఏజెంట్ గత సందర్భాన్ని గుర్తుంచుకొని బహుముఖ సంభాషణను ఛేర్చడానికి సహాయపడుతుంది.

ప్రతి లేయర్‌ను విడిగా క్రమంగా నిర్మిద్దాం.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool డెకొరేటర్‌తో టూల్స్‌ను జోడించడం

టూల్స్ ఏజెంట్‌ను టెక్స్ట్ సృష్టించడం మించి చర్యలు తీసుకోడానికి అనుమతిస్తాయి. `@tool` డెకొరేటర్ సాధారణ Python ఫంక్షన్‌ను ఏజెంట్‌కు పిలవగలవయిన వాటిగా మార్చుతుంది.

ముఖ్యమైన పాయింట్లు:
- ప్రతీ పారామీటర్‌ను మోడల్ అర్థం చేసుకునేందుకు `Annotated[type, "description"]` ఉపయోగించండి.
- డాక్స్ట్రింగ్ మోడల్ చూడగల టూల్ వివరణ అవుతుంది.
- `approval_mode="never_require"` అంటే టూల్ యూజర్ ఆమోదం లేకుండా ఆటోమేటిక్గా నడుస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## పరికరాలతో ఏజెంట్ సృష్టించడం

ఇప్పుడు మనం క్లయింట్, సూచనలు, మరియు పరికరాలను ఏజెంట్‌లో కలపబోతున్నాము. `సూచనలు` వ్యవస్థ ప్రాంప్ట్‌గా పనిచేస్తాయి — అవి ఏజెంట్ వ్యక్తిత్వం మరియు ప్రవర్తనను నిర్వచిస్తాయి.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## సెషన్లతో బహు-తిరుగుడు సంభాషణలు

ఒక `AgentSession` (`agent.create_session()` ద్వారా సృష్టించబడింది) ఒక సంభాషణలో అన్ని సందేశాలను ట్రాక్ చేస్తుంది. ప్రతి `agent.run()` కాల్ కి ఇదే సెషన్ ను అందించడం ద్వారా, ఏజెంట్ కి పూర్తి సంభాషణ చరిత్ర కి ప్రాప్తి ఉంటుంది మరియు గత సందేశాలను తిరిగి చూస్తుంది.

మేము `tools=[check_destination_availability]` ని ఇస్తున్నాము కాబట్టి ఏజెంట్ ప్రతి తిరుగుడులో మా అందుబాటు చెకర్ ని పిలవగలదు.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework యొక్క నాలుగు స్థంభాలను అన్వేషించారు:

| ఆలోచన | మీరు నేర్చుకున్నవి |
|---------|------------------|
| **క్లయింట్** | `FoundryChatClient` క్రెడెన్షియల్ ఆధారిత ధ్రువీకరణతో Azure OpenAI కి కనెక్ట్ అవుతుంది |
| **ఏజెంట్** | `provider.create_agent()` మోడల్ కనెక్షన్‌ను సూచనలు మరియు పేరుతో బండిల్ చేస్తుంది |
| **సాధనాలు** | `@tool` డెకరేటర్ ఏజెంట్ పిలవడానికి పైథాన్ ఫంక్షన్లను వెల్లడిస్తుంది |
| **సెషన్** | `agent.create_session()` బహుళ చలనం అంతటా సంభాషణ చరిత్రను నిర్వహిస్తుంది |

ఈ నిర్మాణ 블ాక్‌లు సహా సహజ సంభాషణలు కలిగి ఉండగల ఏజెంట్లను సృష్టిస్తాయి, బయటి ఫంక్షన్ల కాల్స్ చేయగలవు, మరియు సందర్భం నిర్వహించగలవు — తదుపరి పాఠాలలో మరింత అభివృద్ధి చెందిన ఏజెంటిక్ నమూనాలకు ఇది పునాది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
